# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Configuración inferida desde la imagen (fila, col)
        self.start = (0, 0)

        # Estanterías / paredes (no transitables)
        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
            (0, 2),
        }

        # Estados con piso resbaloso
        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        # Estados terminales con su recompensa
        self.terminal_states = {
            (0, 5): 10.0,   # ENTREGA +10
            (2, 2): 2.0,    # CARGA +2
            (3, 5): -10.0,  # PELIGRO MORTAL -10
        }

        # Peligros no terminales (recompensa -3 al entrar)
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Acciones: UP, DOWN, LEFT, RIGHT
        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

        # Mapeos para desviaciones (índices en self.actions)
        # 0:UP,1:DOWN,2:LEFT,3:RIGHT
        self.left_map = {0:2, 1:3, 2:1, 3:0}
        self.right_map = {0:3, 1:2, 2:0, 3:1}

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        S = []
        for r in range(self.height):
            for c in range(self.width):
                s = (r, c)
                if self.is_valid_state(s):
                    S.append(s)
        return S

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        # cualquier otro estado transitable
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve lista de (next_state, prob).
        Si 'state' es terminal, permanece en el mismo estado con prob 1.
        """
        if not self.is_valid_state(state):
            return [(state, 1.0)]

        if self.is_terminal(state):
            return [(state, 1.0)]

        # identifica índice de la acción
        try:
            a_idx = self.actions.index(action)
        except ValueError:
            raise ValueError("Acción inválida")

        # probabilidades según tipo de piso
        if state in self.slippery_states:
            p_intended = 0.6
            p_left = 0.2
            p_right = 0.2
        else:
            p_intended = 0.9
            p_left = 0.05
            p_right = 0.05

        probs = {}

        # función para aplicar movimiento y comprobar colisiones
        def move(s, act):
            nr, nc = s[0] + act[0], s[1] + act[1]
            ns = (nr, nc)
            if not self.is_valid_state(ns):
                return s
            return ns

        # intención
        intended = move(state, self.actions[a_idx])
        left = move(state, self.actions[self.left_map[a_idx]])
        right = move(state, self.actions[self.right_map[a_idx]])

        probs[intended] = probs.get(intended, 0.0) + p_intended
        probs[left] = probs.get(left, 0.0) + p_left
        probs[right] = probs.get(right, 0.0) + p_right

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 25
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    ev = 0.0
    for ns, p in grid.get_transition_probs(state, action):
        ev += p * V.get(ns, 0.0)
    return ev


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}
    # inicializar valores terminales con su recompensa
    for t, r in grid.terminal_states.items():
        if t in V:
            V[t] = r

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            best = -float('inf')
            for a in grid.actions:
                val = expected_next_value(grid, s, a, V)
                if val > best:
                    best = val

            V_new[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            return V, it

    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        best_a = None
        best_val = -float('inf')
        for a in grid.actions:
            val = expected_next_value(grid, s, a, V)
            if val > best_val:
                best_val = val
                best_a = a
        policy[s] = best_a
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}
    for t, r in grid.terminal_states.items():
        if t in V:
            V[t] = r

    for it in range(1, max_iter + 1):
        delta = 0.0
        for s in S:
            if grid.is_terminal(s):
                continue
            a = policy[s]
            v_new = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < threshold:
            return V
    return V


def policy_improvement(grid, V):
    policy = {}
    stable = True
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        best_a = None
        best_val = -float('inf')
        for a in grid.actions:
            val = expected_next_value(grid, s, a, V)
            if val > best_val:
                best_val = val
                best_a = a
        policy[s] = best_a
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # política inicial aleatoria (accion UP por defecto)
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for it in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)
        history.append(new_policy)
        if all(new_policy.get(s) == policy.get(s) for s in new_policy):
            return new_policy, V, history
        policy = new_policy

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -3.089 |  -3.848 |   WALL   |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.605 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 ↓  |  ←  |  #  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{(0, 0): (-1, 0), (0, 1): (0, -1), (0, 4): (0, 1), (1, 0): (1, 0), (1, 2): (1, 0), (1, 3): (0, 1), (1, 4): (0, 1), (1, 5): (-1, 0), (2, 0): (0, 1), (2, 1): (0, 1), (2, 3): (0, -1), (2, 5): (-1, 0), (3, 0): (0, 1), (3, 1): (0, 1), (3, 2): (-1, 0), (3, 3): (0, -1), (3, 4): (0, -1), (4, 0): (0, 1), (4, 1): (-1, 0), (4, 3): (-1, 0), (4, 4): (0, -1), (4, 5): (0, -1)}, {(0, 0): (1, 0), (0, 1): (0, -1), (0, 4): (0, 1), (1, 0): (1, 0), (1, 2)


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Respuestas (predicciones) y explicaciones

- **1. Desde `START`:** Predigo que el robot intentará la **entrega +10** (más recompensa neta aun considerando el costo por paso), salvo que el costo por paso sea mucho más negativo o que la dinámica resbalosa aumente mucho el riesgo.
- **2. Recompensa menor óptima:** Una recompensa menor (p. ej. `+2`) puede ser preferida si está mucho más cerca: el costo por pasos y el riesgo hacen que la recompensa lejana esperada (descontada y con mayor probabilidad de caer en peligros) sea inferior a la inmediata.
- **3. Estados donde el piso resbaloso cambia la decisión:** Principalmente en celdas adyacentes a peligros (`-3`, `-10`) o en cruces donde una desviación incrementa la probabilidad de terminar en un peligro; p. ej. alrededor de `(1,2)`, `(2,1)`, `(3,3)`.
- **4. Papel del costo por paso `-1`:** Penaliza trayectos largos y favorece rutas más cortas o acciones terminales rápidas; modifica el umbral en que compensa perseguir la recompensa grande.
- **5. Por qué `T(s,a,s')` varía por estado:** Porque algunos suelos son resbalosos (cambian las probabilidades), y las terminales y paredes provocan dinámicas absorbentes o de permanencia; por tanto no es homogénea.

Las celdas siguientes ejecutan los experimentos A/B/C y una búsqueda aproximada del umbral de `living_reward` donde la política desde `START` cambia.

In [7]:
# ---------- Utilidades para experimentar ----------
def evaluate_and_print(grid):
    V_vi, n_vi = value_iteration(grid)
    pi_vi = extract_policy(grid, V_vi)
    print("Iteraciones (VI):", n_vi)
    print_values(grid, V_vi)
    print("Política (VI):")
    print_policy(grid, pi_vi)
    return pi_vi, V_vi


def simulate_policy(grid, policy, start=None, max_steps=100):
    if start is None:
        start = grid.start
    s = start
    visited = set()
    for i in range(max_steps):
        if grid.is_terminal(s):
            return s
        a = policy.get(s)
        if a is None:
            return None
        # determinísticamente aplicar la acción (sin ruido)
        ns = (s[0] + a[0], s[1] + a[1])
        if not grid.is_valid_state(ns):
            ns = s
        if ns in visited:
            return None
        visited.add(ns)
        s = ns
    return None

# función que devuelve la primera meta alcanzada por la política desde START
def policy_goal_from_start(grid, policy):
    term = simulate_policy(grid, policy, start=grid.start)
    return term

print("Definidas utilidades de experimentación.")

Definidas utilidades de experimentación.


In [8]:
# Experimento A — Menos costo por paso
gridA = WarehouseMDP()
gridA.living_reward = -0.1
piA, VA = evaluate_and_print(gridA)
goalA = policy_goal_from_start(gridA, piA)
print("Meta desde START (A):", goalA, "recompensa:", gridA.get_reward(goalA) if goalA else None)

Iteraciones (VI): 26
 +0.767 |  +0.573 |   WALL   |   WALL   |  +8.591 | +10.000
 +0.996 |   WALL   |  +2.892 |  +3.911 |  +4.550 |  +8.591
 +1.243 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874
Política (VI):
 ↓  |  ←  |  #  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Meta desde START (A): (2, 2) recompensa: 2.0


In [9]:
# Experimento B — Piso muy resbaloso (0.40/0.30/0.30)
gridB = WarehouseMDP()
# modificar dinámicamente get_transition_probs: cambiamos probabilidades para estados resbalosos
gridB.slippery_prob = (0.4, 0.3, 0.3)  # (intended,left,right)

# Monkey-patch temporal: reemplazamos el método para usar gridB.slippery_prob
def get_transition_probs_b(state, action):
    if not gridB.is_valid_state(state):
        return [(state, 1.0)]
    if gridB.is_terminal(state):
        return [(state, 1.0)]
    a_idx = gridB.actions.index(action)
    if state in gridB.slippery_states:
        p_intended, p_left, p_right = gridB.slippery_prob
    else:
        p_intended, p_left, p_right = 0.9, 0.05, 0.05

    probs = {}
    def move(s, act):
        nr, nc = s[0] + act[0], s[1] + act[1]
        ns = (nr, nc)
        if not gridB.is_valid_state(ns):
            return s
        return ns

    intended = move(state, gridB.actions[a_idx])
    left = move(state, gridB.actions[gridB.left_map[a_idx]])
    right = move(state, gridB.actions[gridB.right_map[a_idx]])

    probs[intended] = probs.get(intended, 0.0) + p_intended
    probs[left] = probs.get(left, 0.0) + p_left
    probs[right] = probs.get(right, 0.0) + p_right
    return list(probs.items())

gridB.get_transition_probs = get_transition_probs_b

piB, VB = evaluate_and_print(gridB)
goalB = policy_goal_from_start(gridB, piB)
print("Meta desde START (B):", goalB)

Iteraciones (VI): 20
 -3.495 |  -4.209 |   WALL   |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.408 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041
Política (VI):
 ↓  |  ←  |  #  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Meta desde START (B): (2, 2)


In [10]:
# Experimento C — Más paciencia (gamma=0.99)
gridC = WarehouseMDP()
gridC.gamma = 0.99
piC, VC = evaluate_and_print(gridC)
goalC = policy_goal_from_start(gridC, piC)
print("Meta desde START (C):", goalC)

Iteraciones (VI): 23
 -3.324 |  -4.397 |   WALL   |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.297 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892
Política (VI):
 ↓  |  ←  |  #  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Meta desde START (C): (2, 2)


In [11]:
# Bonus: buscar umbral aproximado de living_reward donde la política desde START cambia
def find_threshold(grid_factory, low=-5.0, high=0.0, steps=30):
    lo, hi = low, high
    last_goal = None
    result = []
    for w in np.linspace(lo, hi, steps):
        g = grid_factory()
        g.living_reward = float(w)
        V, _ = value_iteration(g)
        pi = extract_policy(g, V)
        goal = policy_goal_from_start(g, pi)
        result.append((w, goal))
    return result

res = find_threshold(WarehouseMDP, low=-5.0, high=0.0, steps=41)
for w, goal in res:
    print(f"living={w:+.2f} -> goal: {goal}")

print("Hecho: inspecciona la salida para identificar el umbral aproximado.")

living=-5.00 -> goal: (2, 2)
living=-4.88 -> goal: (2, 2)
living=-4.75 -> goal: (2, 2)
living=-4.62 -> goal: (2, 2)
living=-4.50 -> goal: (2, 2)
living=-4.38 -> goal: (2, 2)
living=-4.25 -> goal: (2, 2)
living=-4.12 -> goal: (2, 2)
living=-4.00 -> goal: (2, 2)
living=-3.88 -> goal: (2, 2)
living=-3.75 -> goal: (2, 2)
living=-3.62 -> goal: (2, 2)
living=-3.50 -> goal: (2, 2)
living=-3.38 -> goal: (2, 2)
living=-3.25 -> goal: (2, 2)
living=-3.12 -> goal: (2, 2)
living=-3.00 -> goal: (2, 2)
living=-2.88 -> goal: (2, 2)
living=-2.75 -> goal: (2, 2)
living=-2.62 -> goal: (2, 2)
living=-2.50 -> goal: (2, 2)
living=-2.38 -> goal: (2, 2)
living=-2.25 -> goal: (2, 2)
living=-2.12 -> goal: (2, 2)
living=-2.00 -> goal: (2, 2)
living=-1.88 -> goal: (2, 2)
living=-1.75 -> goal: (2, 2)
living=-1.62 -> goal: (2, 2)
living=-1.50 -> goal: (2, 2)
living=-1.38 -> goal: (2, 2)
living=-1.25 -> goal: (2, 2)
living=-1.12 -> goal: (2, 2)
living=-1.00 -> goal: (2, 2)
living=-0.88 -> goal: (2, 2)
living=-0.75 -